# Combined Extract-Load-Build

This notebook demonstrates the end-to-end document-graph pipeline:

1. **Extract** — Read structured data from CSV
2. **Transform** — Convert rows to typed graph nodes and infer edges
3. **Load/Build** — Write the resulting graph to Neptune
4. **Query** — Read back using the DocumentGraphQueryEngine

This is the simplest path from tabular data to a queryable property graph.

**Prerequisites:**
- `pip install graphrag-toolkit-document-graph[graphrag]` (includes lexical-graph for Neptune connectivity)
- Neptune cluster endpoint (see 00-Setup)


## Configuration

In [ ]:
import os

GRAPH_STORE = os.environ.get(
    'GRAPH_STORE',
    'neptune-db://<your-neptune-cluster-endpoint>:8182'
)
TENANT = 'demo'
CSV_PATH = 'data/users.csv'

print(f'Graph store: {GRAPH_STORE}')
print(f'Tenant:      {TENANT}')
print(f'Data source: {CSV_PATH}')

## Step 1: Extract — Read CSV

Load structured records from a CSV file. Each row becomes a candidate graph node.

In [ ]:
import csv

with open(CSV_PATH) as f:
    records = list(csv.DictReader(f))

print(f'Read {len(records)} records from {CSV_PATH}')
print(f'Columns: {list(records[0].keys())}')
print(f'\nFirst record:')
records[0]

## Step 2: Transform — Row to Node

The `RowToNodeTransformer` converts each tabular row into a typed graph node.
Each record becomes a node with:
- A label (node type) derived from configuration
- An ID from a designated field
- Properties from the row's columns

In [ ]:
from graphrag_toolkit.document_graph.transform.transformer_provider_config import TransformerProviderConfig
from graphrag_toolkit.document_graph.transform.graph_transformers.row_to_node import RowToNodeTransformer

config = TransformerProviderConfig(name='row_to_node', args={'type': 'User'})
row_to_node = RowToNodeTransformer(config)
nodes = row_to_node.transform(records)

print(f'Transformed {len(records)} rows → {len(nodes)} nodes')
print(f'\nFirst node:')
nodes[0]

## Step 3: Transform — Infer Edges

The `EdgeInferencer` examines shared field values across nodes to create relationships.
For example, users sharing the same `account_id` get connected with a `SAME_ACCOUNT` edge.

In [ ]:
from graphrag_toolkit.document_graph.transform.graph_transformers.infer_edges import EdgeInferencer

config = TransformerProviderConfig(
    name='infer_edges',
    args={'source_field': 'account_id', 'edge_type': 'SAME_ACCOUNT'}
)
edge_inferencer = EdgeInferencer(config)
edges = edge_inferencer.transform(records)

print(f'Inferred {len(edges)} edges')
if edges:
    print(f'\nFirst edge:')
    print(f'  {edges[0]}')

## Step 4: Build — Write to Neptune

Generate openCypher MERGE statements and execute against Neptune.
Nodes are tenant-scoped using the shared `TenantId.format_label()` format,
ensuring compatibility with lexical-graph in the same Neptune cluster.

In [ ]:
from graphrag_toolkit.lexical_graph.storage import GraphStoreFactory
from graphrag_toolkit.document_graph import Node
from graphrag_toolkit.document_graph.graph_build.cypher_builder import node_to_cypher

graph_store = GraphStoreFactory.for_graph_store(GRAPH_STORE)
gs = graph_store.__enter__()

# Write nodes
for n in nodes:
    node = Node(
        id=n.get('id', n.get('_id', '')),
        labels=[n.get('node_type', 'Row')],
        properties=n
    )
    cypher, params = node_to_cypher(node, tenant_id=TENANT)
    gs.execute_query(cypher, params)

print(f'✅ Wrote {len(nodes)} nodes to Neptune (tenant={TENANT})')

## Step 5: Query — Read Back

Use `DocumentGraphQueryEngine` to query the tenant-scoped graph.
The engine automatically applies the correct tenant label format.

In [ ]:
from graphrag_toolkit.document_graph.query import DocumentGraphQueryEngine

engine = DocumentGraphQueryEngine(gs, tenant_id=TENANT)
result = engine.get_nodes('User', limit=10)

print(f'Found {len(result)} User nodes in tenant "{TENANT}":')
for r in result:
    print(f'  {r}')

# Clean up
graph_store.__exit__(None, None, None)